# 05 · Workspace — many tenants, one database, isolated

`workspace` partitions storage so tenants share one database safely: the relational stores partition by a `workspace` column and the graph gets a per-workspace name. Two tenants, one `lightrag_tenants` database, distinct fictional corpora.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
from _common.rag import reset_rag
TENANTS = {
  "acme":   ["Acme Corporation makes gadgets in the desert. Wile E. Coyote is Acme's loyal customer.",
             "Wile E. Coyote uses Acme rockets to chase the Road Runner across the canyon."],
  "globex": ["Globex Corporation is led by Hank Scorpio, based in Cypress Creek.",
             "Hank Scorpio gave his employee a house in Cypress Creek near Globex HQ."],
}
PROBE = {"acme": "Wile E. Coyote", "globex": "Hank Scorpio"}
rags = {}
for ws in TENANTS:
    r = build_rag("lightrag_tenants", workspace=ws)
    await r.initialize_storages(); rags[ws] = r
await initialize_pipeline_status()
for ws, r in rags.items():
    print(f"workspace {ws!r} -> graph '{r.chunk_entity_relation_graph.graph_name}'")
    await reset_rag(r)
    await r.ainsert(TENANTS[ws], file_paths=[f"{ws}-{i}" for i in range(len(TENANTS[ws]))])

workspace 'acme' -> graph 'acme_chunk_entity_relation'


workspace 'globex' -> graph 'globex_chunk_entity_relation'


## Each tenant sees only its own entities

In [2]:
labels = {ws: set(await r.chunk_entity_relation_graph.get_all_labels()) for ws, r in rags.items()}
for ws in TENANTS:
    print(f"  {ws}: {sorted(labels[ws])}")
print("  shared:", labels["acme"] & labels["globex"] or "EMPTY (isolated)")

  acme: ['Acme', 'Acme Corporation', 'Canyon', 'Desert', 'Road Runner', 'Wile E. Coyote']
  globex: ['Cypress Creek', 'Globex Corporation', 'Globex HQ', 'Hank Scorpio']
  shared: EMPTY (isolated)


## Cross-tenant probe — graph + retrieval isolation

In [3]:
for ws, r in rags.items():
    other = "globex" if ws == "acme" else "acme"
    has = await r.chunk_entity_relation_graph.has_node(PROBE[other])
    ans = await r.aquery(f"Who is {PROBE[other]}?", QueryParam(mode="mix", enable_rerank=False))
    print(f"  {ws}: has '{PROBE[other]}'? {has}  | answer: {str(ans).strip()[:80]}")

  acme: has 'Hank Scorpio'? False  | answer: I do not have enough information to answer that.


  globex: has 'Wile E. Coyote'? False  | answer: I do not have enough information to answer.


In [4]:
for r in rags.values():
    await r.finalize_storages()